In [1]:
# Part 0: Imports and environment
from dotenv import load_dotenv
import os
import json
import re
from typing import Any
from pathlib import Path

import pandas as pd
import boto3

load_dotenv(override=True)

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
# Hardcode region to N. Virginia for Nova model availability.
AWS_DEFAULT_REGION = "us-east-1"

if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    raise RuntimeError("AWS credentials are missing. Check your .env file.")

print("Environment loaded.")
print(f"Region: {AWS_DEFAULT_REGION}")

Environment loaded.
Region: us-east-1


In [2]:
# Part 1: Create clients and fetch Bedrock catalog
def create_bedrock_session_and_clients():
    """Create AWS session plus Bedrock management/runtime clients."""
    sess = boto3.Session(
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
        aws_session_token=AWS_SESSION_TOKEN,
        region_name=AWS_DEFAULT_REGION,
    )
    reg = sess.region_name or AWS_DEFAULT_REGION or "ap-south-1"
    mgmt = sess.client("bedrock", region_name=reg)
    runtime = sess.client("bedrock-runtime", region_name=reg)
    return sess, reg, mgmt, runtime

def fetch_all_foundation_models(client):
    models = []
    next_token = None
    while True:
        kwargs = {"byOutputModality": "TEXT"}
        if next_token:
            kwargs["nextToken"] = next_token
        resp = client.list_foundation_models(**kwargs)
        models.extend(resp.get("modelSummaries", []))
        next_token = resp.get("nextToken")
        if not next_token:
            break
    return models

session, region, bedrock, bedrock_runtime = create_bedrock_session_and_clients()
foundation_models = fetch_all_foundation_models(bedrock)

print("Bedrock clients initialized successfully")
print(f"Region: {region}")
print(f"Total text models in catalog: {len(foundation_models)}")

Bedrock clients initialized successfully
Region: us-east-1
Total text models in catalog: 93


In [3]:
# Part 2: Use Amazon Nova Lite via Inference Profile
from botocore.exceptions import ClientError

# Use the US inference profile ID recommended by AWS support.
TARGET_INFERENCE_PROFILE_ID = "us.amazon.nova-lite-v1:0"
print(f"Target inference profile ID: {TARGET_INFERENCE_PROFILE_ID}")

chosen_model_id = TARGET_INFERENCE_PROFILE_ID
model_name = "Amazon Nova Lite (US Inference Profile)"
print("Status: USING INFERENCE PROFILE")
print(f"Model name: {model_name}")
print(f"chosen_model_id: {chosen_model_id}")

Target inference profile ID: us.amazon.nova-lite-v1:0
Status: USING INFERENCE PROFILE
Model name: Amazon Nova Lite (US Inference Profile)
chosen_model_id: us.amazon.nova-lite-v1:0


In [4]:
# Part 3: Pick one image and define extraction prompt
image_root = Path(r"C:\Users\Rohit.Sahu\Desktop\experiment\pdf_images\Document_1")
image_exts = {".jpg", ".jpeg", ".png", ".webp"}
image_files = sorted([p for p in image_root.rglob("*") if p.suffix.lower() in image_exts])

if not image_files:
    raise RuntimeError(f"No image files found under {image_root}")

image_path = image_files[0]
ext = image_path.suffix.lower().lstrip(".")
image_format = "jpeg" if ext == "jpg" else ext
if image_format not in {"jpeg", "png", "webp", "gif"}:
    image_format = "jpeg"

with open(image_path, "rb") as f:
    image_bytes = f.read()

print(f"Using image: {image_path}")
print(f"Image format for Bedrock: {image_format}")
print(f"Image bytes: {len(image_bytes)}")

nova_extraction_prompt = """You are a medical document OCR extractor.
Analyze the provided document image and extract information into the JSON schema below.
Return ONLY valid JSON (no markdown, no explanations).

Guidelines:
- Extract visible values only; do not infer or fabricate data
- Leave empty string (\"\") or empty list [] for missing fields
- Preserve text exactly as it appears in the document
- Return all schema keys

"- For every extracted field, provide an approximate bounding box.\n",
"- Bounding box format: [y_min, x_min, y_max, x_max]\n",
"- Coordinates should be normalized between 0 and 1000.\n",
"- If location cannot be determined, return [].\n",
"\n",
Schema:
{
  \"claimant_name\": {\"value\": \"\", \"bbox\": []},\n",
  \"claimant_number\": {\"value\": \"\", \"bbox\": []},\n",
  \"tax_id\": {\"value\": \"\", \"bbox\": []},\n",
  \"practice_address\": {\"value\": \"\", \"bbox\": []},\n",
  \"billing_address\": {\"value\": \"\", \"bbox\": []},\n",
  \"diagnosis_codes\": [],
  \"date_of_service\": {\"value\": \"\", \"bbox\": []},\n",
  \"cpt_codes\": [],
  \"charges\": [],
  \"units\": [],
  \"invoice_date\": {\"value\": \"\", \"bbox\": []},\n",
  \"invoice_number\": {\"value\": \"\", \"bbox\": []},\n",
  \"taxonomy\": {\"value\": \"\", \"bbox\": []},\n",
  \"total_amount\": {\"value\": \"\", \"bbox\": []}\n",
}
"""

print("Prompt ready.")

Using image: C:\Users\Rohit.Sahu\Desktop\experiment\pdf_images\Document_1\page_1.jpg
Image format for Bedrock: jpeg
Image bytes: 253061
Prompt ready.


In [5]:
# Part 4: Invoke Amazon Nova Lite model and parse response
if not chosen_model_id:
    raise RuntimeError("chosen_model_id is not set. Run Part 2 first.")

bedrock_runtime = session.client("bedrock-runtime", region_name=region)
used_nova_model_id = None
nova_raw_response = None
nova_parsed = None

def try_parse_json_payload(raw_text: str) -> Any:
    if not raw_text:
        return None
    try:
        return json.loads(raw_text)
    except Exception:
        pass
    start = raw_text.find("{")
    end = raw_text.rfind("}")
    if start != -1 and end != -1 and end > start:
        chunk = raw_text[start:end + 1]
        try:
            return json.loads(chunk)
        except Exception:
            return None
    return None

try:
    resp = bedrock_runtime.converse(
        modelId=chosen_model_id,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "image": {
                            "format": image_format,
                            "source": {"bytes": image_bytes},
                        }
                    },
                    {"text": nova_extraction_prompt},
                ],
            }
        ],
        inferenceConfig={"maxTokens": 1500, "temperature": 0},
    )

    parts = resp.get("output", {}).get("message", {}).get("content", [])
    text_parts = [
        part.get("text", "")
        for part in parts
        if isinstance(part, dict) and "text" in part
    ]
    nova_raw_response = "\n".join(t for t in text_parts if t).strip()
    used_nova_model_id = chosen_model_id
    nova_parsed = try_parse_json_payload(nova_raw_response)

except ClientError as e:
    code = e.response.get("Error", {}).get("Code", "Unknown")
    msg = e.response.get("Error", {}).get("Message", str(e))
    raise RuntimeError(f"{chosen_model_id} -> {code}: {msg}") from e
except Exception as e:
    raise RuntimeError(f"{chosen_model_id} -> {str(e)}") from e

print(f"Used Nova model: {used_nova_model_id}")
print("Raw response preview:")
print((nova_raw_response or "")[:1200])

if nova_parsed is not None:
    print("\nParsed JSON:")
    print(json.dumps(nova_parsed, indent=2, ensure_ascii=False))
else:
    print("\nCould not parse strict JSON; keeping raw response in nova_raw_response.")

Used Nova model: us.amazon.nova-lite-v1:0
Raw response preview:
```json
{
  "claimant_name": "Maria Mcclure",
  "claimant_number": "4014741972",
  "tax_id": "AAA",
  "practice_address": "13093 CANOBURY ST DETROIT MI 48205",
  "billing_address": "",
  "diagnosis_codes": [],
  "date_of_service": "08/07/2025",
  "cpt_codes": [],
  "charges": ["2,976.35", "2,976.35"],
  "units": ["1", "1"],
  "invoice_date": "08/07/2025",
  "invoice_number": "202510140011526",
  "taxonomy": "10142025",
  "total_amount": "5,952.70"
}
```

Parsed JSON:
{
  "claimant_name": "Maria Mcclure",
  "claimant_number": "4014741972",
  "tax_id": "AAA",
  "practice_address": "13093 CANOBURY ST DETROIT MI 48205",
  "billing_address": "",
  "diagnosis_codes": [],
  "date_of_service": "08/07/2025",
  "cpt_codes": [],
  "charges": [
    "2,976.35",
    "2,976.35"
  ],
  "units": [
    "1",
    "1"
  ],
  "invoice_date": "08/07/2025",
  "invoice_number": "202510140011526",
  "taxonomy": "10142025",
  "total_amount": "5,952.

In [6]:
# Part 5: Save model extraction result
output_dir = Path(r"C:\Users\Rohit.Sahu\Desktop\experiment\extraction_results")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "nova_lite_single_image_extraction.json"
result_to_save = nova_parsed if nova_parsed is not None else {"raw_response": nova_raw_response}

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(result_to_save, f, indent=2, ensure_ascii=False)

print(f"Saved extraction output to: {output_file}")

Saved extraction output to: C:\Users\Rohit.Sahu\Desktop\experiment\extraction_results\nova_lite_single_image_extraction.json


In [ ]:
from PIL import Image, ImageDraw, ImageOps

if nova_parsed is not None:
    try:
        img = Image.open(image_path)
        img = ImageOps.exif_transpose(img)
        draw = ImageDraw.Draw(img)
        
        img_w, img_h = img.size
        
        for field, data in nova_parsed.items():
            if not isinstance(data, dict):
                continue
            
            bbox = data.get("bbox", [])
            if len(bbox) != 4:
                continue
            
            y1 = bbox[0] * img_h / 1000
            x1 = bbox[1] * img_w / 1000
            y2 = bbox[2] * img_h / 1000
            x2 = bbox[3] * img_w / 1000
            
            draw.rectangle(
                [x1, y1, x2, y2],
                outline="red",
                width=4
            )
            
            draw.text(
                (x1, max(0, y1 - 20)),
                field,
                fill="red"
            )
        
        out_filename = "highlighted_" + Path(image_path).stem + ".png"
        img.save(out_filename)
        img.show()
        print(f"Saved highlighted image to {out_filename}")
    except Exception as e:
        print("Error plotting bounding boxes:", e)
else:
    print("No parsed JSON to draw bounding boxes from.")
